In [1]:
import sys

sys.path.append("../src")

In [2]:
import os
import sys
from dotenv import load_dotenv

# 1. Pugem un nivell per anar a l'arrel on hi ha el .env
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

# 2. Carreguem el .env explícitament
load_dotenv()

# 3. Debug per confirmar que ara sí que el llegeix
print(f"Directori actual: {os.getcwd()}")
print(f"URL llegida: {os.getenv('DB_URL_DEVELOPMENT')}") # Hauria de sortir la ruta de la BD

# 4. Afegim src al path per trobar el mòdul domain
if "src" not in sys.path:
    sys.path.append("src")

# 5. ARA sí, fem les importacions
from sqlalchemy import select, and_, or_, func
from domain import (
    Base, engine, Session, Employee, AppCredential,
    Category, Allergen, Dish, Menu, Sale
)

print("🚀 Tot carregat correctament!")

Directori actual: C:\Users\marcc\source\repos\restaurant_backoffice
URL llegida: sqlite:///notebooks/restaurant_backoffice.db
🚀 Tot carregat correctament!


In [13]:
### Simple query ###
with Session() as session: #Session és l'equivalent una transacció a una base de dades. Crea la connexió amb base de dades per fer la transacció.
 #With empaqueta l'acció de l'objecte Session() i fa per defecte rollback sinò afegim nosaltres el commit()
    stmt = select(Menu) #Creem objecte tipus statement+
    menus = session.scalars(stmt) #Scalars torna una tupla de, en aquest cas menú

    for menu in menus:
        print(f"Name: {menu.menu_name}")

Name: Menú cap de setmana
Name: Menú Nit
Name: Menú Diari


In [14]:
### Simple query ### --> Order by
with Session() as session: #Session és l'equivalent una transacció a una base de dades. Crea la connexió amb base de dades per fer la transacció.
 #With empaqueta l'acció de l'objecte Session() i fa per defecte rollback sinò afegim nosaltres el commit()
    stmt = select(Menu).order_by(Menu.menu_name.desc(), Menu.price) #Creem objecte tipus statement+
    menus = session.scalars(stmt) #Scalars torna una tupla de, en aquest cas menú

    for menu in menus:
        print(f"Name: {menu.menu_name}")

Name: Menú cap de setmana
Name: Menú Nit
Name: Menú Diari


In [16]:
### Simple query ### --> Where clause 1
with Session() as session: #Session és l'equivalent una transacció a una base de dades. Crea la connexió amb base de dades per fer la transacció.
 #With empaqueta l'acció de l'objecte Session() i fa per defecte rollback sinò afegim nosaltres el commit()
    stmt = select(Menu).where(Menu.price < 15).order_by(Menu.menu_name) #Creem objecte tipus statement+
    menus = session.scalars(stmt) #Scalars torna una tupla de, en aquest cas menú

    for menu in menus:
        print(f"Name: {menu.menu_name}")

Name: Menú Diari
Name: Menú cap de setmana


In [17]:
### Simple query ### --> Where clause 2 ¡¡¡¡ Acabar aplicant el and or !!!!
with (Session() as session): #Session és l'equivalent una transacció a una base de dades. Crea la connexió amb base de dades per fer la transacció.
 #With empaqueta l'acció de l'objecte Session() i fa per defecte rollback sinò afegim nosaltres el commit()
    stmt = (
        select(Menu)
        .where(Menu.price < 15) #Creem objecte tipus statement+

        ) .order_by(Menu.menu_name)

    menus = session.scalars(stmt) #Scalars torna una tupla de, en aquest cas menú

    for menu in menus:
        print(f"Name: {menu.menu_name}")

Name: Menú Diari
Name: Menú cap de setmana


In [23]:
### Simple query ### --> Un resultat o res
with (Session() as session):
    stmt = select(Menu).where(Menu.id==1)
    menu = session.scalar(stmt) #Scalar en lloc de scallars, tornarà un registre

    if menu:
        print(f"Name: {menu.menu_name}")

Name: Menú Diari


In [3]:
### Simple query ### --> Get by id (com un resultat però més ràpid) --> Fem join directament aquí
with (Session() as session):
    menu = session.get(Menu, 1) #Recuperem menú amb id 1

    if menu:
        print(f"Name: {menu.menu_name}")
        ### Joint amb plats
        print("Plats")
        for dish in menu.dishes:
            print(f"    {dish.dish_name}")

Name: Menú Diari
Plats
    Amanida Cèsar
    Macarrons a la bolonyesa
    Aigua mineral


In [4]:
### Insert ###
with Session() as session:
    # 1. Creem el menú
    weekend_menu=Menu()
    weekend_menu.menu_name="Menú cap de setmana"
    weekend_menu.price=12.50

    # 2. Busquem els objectes Dish que coincideixin amb els noms
    new_dishes = ["Amanida Cèsar", "Macarrons a la bolonyesa", "Aigua mineral"]

    # Fem una consulta per portar els objectes Dish de la base de dades
    query = select(Dish).where(Dish.dish_name.in_(new_dishes))
    dishes_objects = session.scalars(query).all()

    # 3. Assignem la LLISTA d'objectes (no el string!)
    weekend_menu.dishes = list(dishes_objects)

    session.add(weekend_menu)
    session.commit()

In [6]:
### Filtres avançats, exemple or ###

#prèviament importar or des de sqlalchemy
with Session() as session:
    #Busquem plats que siguin "Begudes" o "Postres"
    #Necessitem id categoria
    #stmt (statement/declaració) és la variable que contindrà la consulta.
    stmt = select(Dish).where(
        or_ (
            Dish.id_category==3, Dish.id_category == 4
        )
    )
    results=session.scalars(stmt).all() #results és la variable que guardarà el retorn de la consulta en una espècie de tupla.

    for dish in results:
        print(f"Plat: {dish.dish_name} (Cat ID: {dish.id_category})")

    #Concepte: where() permet filtrar, i or_() agrupa diverses condicions on només cal que una sigui certa.

Plat: Pastís de formatge (Cat ID: 3)
Plat: Aigua mineral (Cat ID: 4)


In [7]:
### Ordenació múltiple ###

#pots ordenar per diversos criteris a la vegada
with Session() as session:
    #Ordenem primer per preu (descendent) i després per nom (ascendent)
    stmt = select(Menu).order_by(
        Menu.price.desc(),
        Menu.menu_name.asc()
    )

    for m in session.scalars(stmt):
        print(f"{m.price}€ - {m.menu_name}")

18.00€ - Menú Nit
12.50€ - Menú Diari
12.50€ - Menú cap de setmana
12.50€ - Menú cap de setmana


In [8]:
### Agregacions i Group by(execute) ###

#Quan no demanes objectes sencers, sinó dades calculades, has d'usar execute() en lloc de scalars().
#Objectes sencers:
#Dades calculades:

#importar prèviament func a sqlalchemy.
with Session() as session:
    #Comptem quants plats hi ha per cada categoria
    stmt=select(
        Dish.id_category,
        func.count(Dish.id).label("total_plats")
    ).group_by(Dish.id_category)

    results = session.execute(stmt).all()

    for row in results:
        print(f"Cateogira ID {row.id_category}: {row.total_plats} plats")

#func permet usar funcions SQL com count, avg o sum.

Cateogira ID 1: 1 plats
Cateogira ID 2: 2 plats
Cateogira ID 3: 1 plats
Cateogira ID 4: 1 plats


In [9]:
### Clàusula having ###

#S'utilitza per filtrar els resultats d'un group_by
with Session() as session:
    # Volem les categories que tinguin més de 2 plats
    stmt = select(
        Dish.id_category,
        func.count(Dish.id)
    ).group_by(Dish.id_category).having(func.count(Dish.id) > 2)

    results = session.execute(stmt).all()
    for row in results:
        print(f"Categoria amb molts plats: ID {row[0]}")

In [10]:
### Paginació limit i offset

#útil on no vols carregar tota la llista de cop.

with Session() as session:
    # Volem la "segona pàgina" de plats, de 2 en 2
    stmt = select(Dish).limit(2).offset(2)

    for dish in session.scalars(stmt):
        print(f"Plat paginat: {dish.dish_name}")

#limit diu quants registres vols, i offset quants n'has de saltar.

Plat paginat: Lluç al forn
Plat paginat: Pastís de formatge


In [11]:
### mappings() ###

#PDF recomana mappings() per rebre els resultats com a diccionaris de Python en lloc de tuples.

with Session() as session:
    stmt = select(Menu.menu_name, Menu.price)
    # Convertim cada fila en un diccionari {'menu_name': ..., 'price': ...}
    results = session.execute(stmt).mappings().all()

    for row in results:
        print(f"Diccionari: {row['menu_name']} costa {row['price']}€")

Diccionari: Menú Diari costa 12.50€
Diccionari: Menú Nit costa 18.00€
Diccionari: Menú cap de setmana costa 12.50€
Diccionari: Menú cap de setmana costa 12.50€
